# Conv1D autoencoder baseline

This notebook is a validation-only experiment dashboard. Reusable preparation, modeling, training, metrics, checkpointing, evaluation, and export live in `Code/iaflow`. The configured model compresses each `log10(A_theta)` surface from `(31, 101)` to two latent variables without skip connections. The test split is not read anywhere in this notebook.

In [ ]:
import json
import sys
from pathlib import Path

import h5py
import numpy as np
import torch
from matplotlib import pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Run this notebook from within the IAFlowCloud repository.')
    PROJECT_ROOT = PROJECT_ROOT.parent
code_path = PROJECT_ROOT / 'Code'
if str(code_path) not in sys.path:
    sys.path.insert(0, str(code_path))

from iaflow.artifacts import load_compatible_autoencoder_checkpoint
from iaflow.config import load_experiment_config
from iaflow.data import CACHE_FORMAT_VERSION, CachedSurfaceDataset, validate_surface_cache
from iaflow.architectures import build_autoencoder

CONFIG_PATH = PROJECT_ROOT / 'Config' / 'NLA' / 'AutoEncoderConv1D.yml'
config = load_experiment_config(CONFIG_PATH, project_root=PROJECT_ROOT)
print('PyTorch:', torch.__version__)
print('Project:', PROJECT_ROOT)
print('Configured latent dimension:', config.model.latent_dim)

## 1. Validate the source-ordered cache

Preparation writes one memory-mappable source-ordered array, not one copy per split. The HDF5 split indices remain authoritative and normalization is estimated only from its training rows.

In [ ]:
cache_directory = config.resolve_path(config.data.cache_directory)
metadata_path = cache_directory / 'Metadata.json'
if metadata_path.exists():
    metadata = validate_surface_cache(config)
    print(json.dumps(metadata, indent=2))
else:
    metadata = None
    print('Cache not prepared. Run:')
    print(f'python -m iaflow.scripts.prepare_data --config {CONFIG_PATH}')

## 2. Inspect the configured bottleneck

The 31 redshift values are channels and convolution runs along the 101-point wavenumber axis. Every reconstruction passes through the two-number bottleneck.

In [ ]:
model = build_autoencoder(config.model, config.data.input_shape)
print(json.dumps(model.architecture_summary(), indent=2))
try:
    from torchinfo import summary
    summary(model, input_size=(2, *config.data.input_shape), device='cpu')
except ImportError:
    print(model)

## 3. Train from the command line

Long jobs run through the script so they produce complete artifacts and can resume independently of the notebook UI.

In [ ]:
print(f'python -m iaflow.scripts.train_autoencoder --config {CONFIG_PATH}')
print('Validation-only smoke option: add --epochs 2 --maximum-train-samples 1024 --maximum-validation-samples 256')

## 4. Review validation history

In [ ]:
latest_pointer = config.resolve_path(config.output.root_directory) / 'LatestRun.txt'
RUN_DIRECTORY = None
if latest_pointer.exists():
    RUN_DIRECTORY = config.resolve_path(latest_pointer.read_text().strip())
if RUN_DIRECTORY is None or not (RUN_DIRECTORY / 'Best.pt').is_file():
    RUN_DIRECTORY = None
    print('No completed compatible run is registered yet.')
else:
    history = json.loads((RUN_DIRECTORY / 'History.json').read_text())
    epochs = [row['epoch'] for row in history]
    train_loss = [row['train_loss'] for row in history]
    validation_loss = [row['validation']['normalized_mse'] for row in history]
    validation_variance = [row['validation']['variance_recovered'] for row in history]
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].semilogy(epochs, train_loss, label='training objective')
    axes[0].semilogy(epochs, validation_loss, label='validation MSE')
    axes[0].set(xlabel='epoch', ylabel='normalized loss')
    axes[0].legend()
    axes[1].plot(epochs, validation_variance)
    axes[1].axhline(0.999, color='black', linestyle='--', label='99.9% target')
    axes[1].set(xlabel='epoch', ylabel='validation variance recovered')
    axes[1].legend()
    figure.tight_layout()
    plt.show()

## 5. Representative and worst-case validation reconstructions

The following diagnostics inspect only a deterministic prefix of the validation split and display its best, median, and worst reconstruction by normalized MSE.

In [ ]:
diagnostic_count = 512
validation_data = None
validation_latent = None
if RUN_DIRECTORY is not None:
    trained_model, normalization, checkpoint, _ = load_compatible_autoencoder_checkpoint(
        RUN_DIRECTORY / 'Best.pt', config
    )
    validation_data = CachedSurfaceDataset(config, 'validation')
    diagnostic_count = min(diagnostic_count, len(validation_data))
    targets = torch.stack([validation_data[index] for index in range(diagnostic_count)])
    with torch.inference_mode():
        validation_latent = trained_model.encode(targets).cpu().numpy()
        reconstructions = trained_model(targets).cpu().numpy()
    target_values = targets.numpy()
    surface_mse = np.mean((reconstructions - target_values) ** 2, axis=(1, 2))
    order = np.argsort(surface_mse)
    selected = [order[0], order[len(order) // 2], order[-1]]
    labels = ['best', 'median', 'worst']
    figure, axes = plt.subplots(3, 3, figsize=(13, 11), constrained_layout=True)
    for row, (index, label) in enumerate(zip(selected, labels)):
        truth = normalization.denormalize(target_values[index])
        prediction = normalization.denormalize(reconstructions[index])
        residual = prediction - truth
        limit = max(float(np.max(np.abs(residual))), np.finfo(np.float32).eps)
        images = [
            axes[row, 0].imshow(truth, aspect='auto', origin='lower'),
            axes[row, 1].imshow(prediction, aspect='auto', origin='lower'),
            axes[row, 2].imshow(residual, aspect='auto', origin='lower', cmap='coolwarm', vmin=-limit, vmax=limit),
        ]
        axes[row, 0].set_title(f'{label}: truth log10(A_theta)')
        axes[row, 1].set_title(f'reconstruction; MSE={surface_mse[index]:.3e}')
        axes[row, 2].set_title('residual')
        for axis, image in zip(axes[row], images):
            axis.set(xlabel='wavenumber index', ylabel='redshift channel')
            figure.colorbar(image, ax=axis, shrink=0.75)
    plt.show()

## 6. Latent geometry and nuisance correlations

These plots show whether the learned two-dimensional manifold has strong non-Gaussian structure and how each coordinate correlates with the 13 original NLA nuisance parameters.

In [ ]:
if validation_latent is not None:
    figure, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].scatter(validation_latent[:, 0], validation_latent[:, 1], s=8, alpha=0.45)
    axes[0].set(xlabel='latent 1', ylabel='latent 2', title='validation latent manifold')
    axes[1].hist(validation_latent[:, 0], bins=35, density=True)
    axes[1].set(xlabel='latent 1', ylabel='density')
    axes[2].hist(validation_latent[:, 1], bins=35, density=True)
    axes[2].set(xlabel='latent 2', ylabel='density')
    figure.tight_layout()
    plt.show()

    source_path = config.resolve_path(config.data.source_path)
    source_indices = validation_data.source_indices[:diagnostic_count]
    with h5py.File(source_path, 'r') as source:
        nuisance_values = source['parameters/values'][source_indices]
        nuisance_names = [
            value.decode() if isinstance(value, bytes) else str(value)
            for value in source['parameters/names'][:]
        ]
    joined = np.column_stack([validation_latent, nuisance_values])
    correlations = np.nan_to_num(np.corrcoef(joined, rowvar=False)[:2, 2:])
    figure, axis = plt.subplots(figsize=(12, 3.2))
    image = axis.imshow(correlations, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
    axis.set(yticks=[0, 1], yticklabels=['latent 1', 'latent 2'])
    axis.set(xticks=np.arange(len(nuisance_names)), xticklabels=nuisance_names)
    axis.tick_params(axis='x', rotation=55)
    figure.colorbar(image, ax=axis, label='Pearson correlation')
    figure.tight_layout()
    plt.show()

## 7. Identical-split PCA comparison

`PCA.ipynb` can export validation metrics for ranks 1, 2, and 10 using the same `log10` target, training mean, and stored validation indices. This notebook reads that compact benchmark rather than loading scikit-learn into MLConda.

In [ ]:
pca_metrics_path = PROJECT_ROOT / 'Data' / 'NLA' / 'PCA' / 'PCAValidationMetrics.json'
if RUN_DIRECTORY is not None and pca_metrics_path.exists():
    pca_metrics = json.loads(pca_metrics_path.read_text())
    run_summary = json.loads((RUN_DIRECTORY / 'Summary.json').read_text())
    rows = [
        ('PCA', int(rank), metrics['variance_recovered'])
        for rank, metrics in sorted(pca_metrics['ranks'].items(), key=lambda item: int(item[0]))
    ]
    rows.append(('Conv1D autoencoder', config.model.latent_dim, run_summary['best_validation_metrics']['variance_recovered']))
    print(f"{'Method':24s} {'Latent size':>12s} {'Validation variance recovered':>30s}")
    print('-' * 70)
    for method, rank, variance in rows:
        print(f'{method:24s} {rank:12d} {variance:30.8%}')
else:
    print('Run PCA.ipynb with EXPORT_VALIDATION_BENCHMARK=True to create the comparison file.')

## 8. Freeze, test once, and export latents

After all choices are frozen, run the final test command once with `--confirm-final-test`. A smoke run must use `--split validation` instead.

```bash
python -m iaflow.scripts.evaluate_autoencoder --config Config/NLA/AutoEncoderConv1D.yml --checkpoint Runs/NLA/AutoEncoder/Conv1D/<final-run>/Best.pt --split test --confirm-final-test
python -m iaflow.scripts.export_latents --config Config/NLA/AutoEncoderConv1D.yml --checkpoint Runs/NLA/AutoEncoder/Conv1D/<final-run>/Best.pt --include-test
```

## Final consistency checks

In [ ]:
assert config.model.name == 'Conv1D'
assert config.model.latent_dim == 2
assert config.data.input_shape == (31, 101)
assert CONFIG_PATH.is_file()
if metadata is not None:
    assert metadata['cache_format_version'] == CACHE_FORMAT_VERSION
    assert tuple(metadata['input_shape']) == config.data.input_shape
    assert 'split_indices' not in metadata
if RUN_DIRECTORY is not None:
    assert not json.loads((RUN_DIRECTORY / 'Summary.json').read_text())['test_split_used_during_training']
print('Autoencoder notebook consistency checks passed.')